# 04 - MLflow Judges and LLM Evaluation

**UI tab:** Evaluation runs

MLflow 3.x evaluation uses:
- `@scorer` decorator to define a judge function
- `Feedback` object to return value + rationale
- `mlflow.genai.evaluate()` to run evaluation

> Start the MLflow server first: `mlflow server --host 127.0.0.1 --port 5000`

In [ ]:
!pip install mlflow google-genai --quiet

In [ ]:
import os
import mlflow
from google import genai
from google.genai import types
from mlflow.genai.scorers import scorer
from mlflow.entities import Feedback

os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY_HERE"
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("04-MLflow-Judges")
print("MLflow", mlflow.__version__, "ready")

## Step 1 - Evaluation dataset with outputs already included

In [ ]:
# Generate predictions first, then put them in the dataset
# This way mlflow.genai.evaluate() does not need a predict_fn
questions = [
    {"question": "What is MLflow?",          "ground_truth": "MLflow is an open-source platform for managing the ML lifecycle."},
    {"question": "What is experiment tracking?", "ground_truth": "Experiment tracking records parameters, metrics and outputs for reproducibility."},
    {"question": "What is a model registry?",  "ground_truth": "A model registry is a central store for versioning and managing ML models."},
]

eval_dataset = []
for row in questions:
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[row["question"]],
        config=types.GenerateContentConfig(system_instruction="Answer in 1-2 sentences.")
    )
    eval_dataset.append({
        "inputs":  {"question": row["question"]},
        "outputs": (response.text or "").strip(),
        "expectations": {"expected_response": row["ground_truth"]},
    })

for row in eval_dataset:
    print(f"Q: {row['inputs']['question']}")
    print(f"A: {row['outputs']}
")

## Step 2 - Define judges using @scorer + Feedback

Exact pattern from the MLflow UI docs.

In [ ]:
# Judge 1: Gemini Pro checks factual correctness
@scorer
def correctness_judge(inputs, outputs, expectations) -> Feedback:
    question     = inputs.get("question", "")
    answer       = str(outputs or "")
    ground_truth = expectations.get("expected_response", "")
    prompt = (
        "Is this answer factually correct?
"
        f"Question: {question}
"
        f"Correct answer: {ground_truth}
"
        f"Given answer: {answer}
"
        "Reply: true or false, then one sentence reason."
    )
    response = client.models.generate_content(
        model="gemini-2.5-pro",
        contents=[prompt],
        config=types.GenerateContentConfig(temperature=0.0, max_output_tokens=50)
    )
    text = (response.text or "").strip()
    return Feedback(value=text.lower().startswith("true"), rationale=text)


# Judge 2: Simple code-based, no LLM needed
@scorer
def conciseness_judge(outputs) -> Feedback:
    word_count = len(str(outputs or "").split())
    is_concise = word_count <= 30
    label      = "concise" if is_concise else "too long"
    return Feedback(value=is_concise, rationale=f"{word_count} words - {label}")

print("Judges defined: correctness_judge, conciseness_judge")

## Step 3 - Run evaluation

Passes dataset through both judges. Results appear in **Evaluation runs** tab.

In [ ]:
# Judge 1: Gemini Pro checks factual correctness
@scorer
def correctness_judge(inputs, outputs, expectations) -> Feedback:
    question     = inputs.get("question", "")
    answer       = str(outputs or "")
    ground_truth = expectations.get("expected_response", "")

    prompt = (
        "Is this answer factually correct?
"
        f"Question: {question}
"
        f"Correct answer: {ground_truth}
"
        f"Given answer: {answer}
"
        "Reply: true or false, then one sentence reason."
    )
    response = client.models.generate_content(
        model="gemini-2.5-pro",
        contents=[prompt],
        config=types.GenerateContentConfig(temperature=0.0, max_output_tokens=50)
    )
    text = (response.text or "").strip()
    is_correct = text.lower().startswith("true")
    return Feedback(value=is_correct, rationale=text)


# Judge 2: Simple code-based, no LLM needed
@scorer
def conciseness_judge(outputs) -> Feedback:
    word_count = len(str(outputs or "").split())
    is_concise = word_count <= 30
    return Feedback(
        value=is_concise,
        rationale=f"{word_count} words - {chr(39)}concise{chr(39)} if is_concise else {chr(39)}too long{chr(39)}",
    )

print("Judges defined: correctness_judge, conciseness_judge")

## Step 4 - Register Judge via AI Gateway (populates Judges tab)

In [ ]:
from mlflow.genai.scorers import Correctness, ScorerSamplingConfig

# Uses the gateway endpoint you created in AI Gateway UI
# endpoint name: gemini-judge, model: gemini-2.5-flash-lite
correctness = Correctness(model="gateway:/gemini-judge")

# register() adds it to the Judges tab
# start() activates it to auto-score every new trace
registered = correctness.register(name="gemini_correctness")
registered.start(sampling_config=ScorerSamplingConfig(sample_rate=1.0))

print("Judge registered and active")
print("Refresh the Judges tab — gemini_correctness should appear")

## MLflow UI - What to explore

**Next** 05_mlflow_datasets.ipynb